In [1]:
import os
import subprocess
import io

for item in os.listdir("../DataLifting_output"):
    try:
        print(f"Loading {item}...")
        
        requete_sql = f"ld_dir('/usr/share/proj', '{item}', 'http://ns.inria.fr/movida/graph'); rdf_loader_run();"
        
        commande = [
            "docker", "exec", "virtuoso-server", 
            "isql", "-U", "dba", "-P", "mysecret", 
            f"exec={requete_sql}"
        ]
        
        resultat = subprocess.run(commande, capture_output=True, text=True)
    
        if resultat.returncode == 0:
            print(f"{item} loaded !")
        else:
            print(f"Error while loading {item} file :")
            print(resultat.stderr)
    except OSError as e:
        print(f"Error:{ e.strerror}")





Loading indicator.ttl...
indicator.ttl loaded !
Loading record.ttl...
record.ttl loaded !
Loading record1.ttl...
record1.ttl loaded !
Loading record2.ttl...
record2.ttl loaded !
Loading record3.ttl...
record3.ttl loaded !
Loading record4.ttl...
record4.ttl loaded !
Loading record5.ttl...
record5.ttl loaded !
Loading record6.ttl...
record6.ttl loaded !
Loading record7.ttl...
record7.ttl loaded !
Loading record8.ttl...
record8.ttl loaded !
Loading space.ttl...
space.ttl loaded !


In [8]:
prefixes = {
    "http://ns.inria.fr/movida/ontology#": "mvdo:",
    "http://ns.inria.fr/movida/thesaurus#": "mvdth:",
    "http://ns.inria.fr/movida/data#": ":",
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#": "rdf:",
    "http://www.w3.org/2000/01/rdf-schema#": "rdfs:",
    "http://www.w3.org/2002/07/owl#": "owl:",
    "http://www.w3.org/2001/XMLSchema#": "xsd:",
    "http://www.w3.org/2004/02/skos/core#": "skos:",
    "http://www.w3.org/2006/time#": "time:",
    "http://www.opengis.net/ont/geosparql#": "geo:",
    "http://www.w3.org/ns/prov#": "prov:",
    "http://qudt.org/schema/qudt/": "qudt:",
    "https://qudt.org/3.3.0/vocab/quantitykind": "quantitykind:",
    "http://www.w3.org/ns/sosa/": "sosa:"
}

def shortenURI(uri):
    if not isinstance(uri, str):
        return uri
    for completeURI, prefix in prefixes.items():
        if uri.startswith(completeURI):
            return uri.replace(completeURI, prefix)
    return uri

In [11]:
import pandas as pd
import requests

def executeSparqlQuery(sparqlQuery, url="http://localhost:8890/sparql"):

    parametres = {
        "query": sparqlQuery,
        "format": "text/csv"
    }

    print("Executing request.")
    reponse = requests.get(url, params=parametres)
    
    df = pd.read_csv(io.StringIO(reponse.text))

    for colonne in df.columns:
        df[colonne] = df[colonne].apply(shortenURI)

    print(f"query answer is {len(df)} item{'s' if len(df)>1 else ''} long")
    return df



In [12]:
url = "http://localhost:8890/sparql"
query = """
    PREFIX mvdo: <http://ns.inria.fr/movida/ontology#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?indicator ?predicate ?objet
    FROM <http://ns.inria.fr/movida/graph>
    WHERE { 
      ?indicator mvdo:hasTheme ?predicate .
      ?indicator rdf:type mvdo:TerritoryIndicator .
    }
"""

df = executeSparqlQuery(query)
display(df)

Executing request.
query answer is 6 items long


,indicator,predicate,objet
0,:indicator-419d0de5cfbcc949292c69c284bef64186a...,mvdth:TransportationMode,NaN
1,:indicator-45d3a321391fdbdd2345bab4ad508f8499c...,mvdth:TripPurpose,NaN
2,:indicator-52b7a26e44f3fa54798e0bdeddfdfac876b...,mvdth:TransportationMode,NaN
3,:indicator-6dc43c05fb6dbb064725d5cf0a831d6a8b1...,mvdth:TripPurpose,NaN
4,:indicator-b0dfa41c6712c9357209cc8deec74520710...,mvdth:TripPurpose,NaN
5,:indicator-b7e1a55ec193edca6ddc632997a63bcd075...,mvdth:TransportationMode,NaN
